# DreamBooth Fine-Tuning (LoRA)
Fine-tune Stable Diffusion v1.5 on a subject using DreamBooth + LoRA with prior-preservation loss.

In [14]:
!pip install -q diffusers==0.27.2 transformers accelerate torchvision tqdm

In [15]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype  = torch.float16 if device == "cuda" else torch.float32
print(f"Device: {device}  |  dtype: {dtype}")
if device == "cpu":
    print("WARNING: No GPU. Enable one via Runtime > Change runtime type > T4 GPU.")

Device: cuda  |  dtype: torch.float16


## Upload source files
Select all 5 files from your local `code/` folder: `model.py`, `data.py`, `train.py`, `inference.py`, `generate_prior.py`.

In [16]:
from google.colab import files
uploaded = files.upload()
for fname in uploaded:
    print(f"loaded {fname}")

KeyboardInterrupt: 

## Upload instance images

In [9]:
import os
from google.colab import files

INSTANCE_DIR = "data/killian"
os.makedirs(INSTANCE_DIR, exist_ok=True)

uploaded = files.upload()
for fname, data in uploaded.items():
    with open(os.path.join(INSTANCE_DIR, fname), "wb") as f:
        f.write(data)

print(f"Uploaded {len(uploaded)} images to {INSTANCE_DIR}")

KeyboardInterrupt: 

## Generate prior images (~15 min on T4)

In [ ]:
!pip install -q diffusers==0.27.2 transformers accelerate torchvision tqdm huggingface_hub==0.23.4

In [ ]:
from generate_prior import generate_prior_images

PRIOR_DIR = "data/prior/person"

generate_prior_images(
    class_prompt="a person",
    output_dir=PRIOR_DIR,
    num_images=200,
    device=device,
    dtype=dtype,
)

## Train (~30-40 min on T4)

In [ ]:
from train import training_loop

training_loop(
    instance_dir=INSTANCE_DIR,
    class_dir=PRIOR_DIR,
    instance_prompt="a sks person",
    class_prompt="a person",
    output_dir="checkpoints",
    validation_dir="validation",
    num_steps=800,
    device=device,
    dtype=dtype,
)

## Run inference

In [ ]:
import matplotlib.pyplot as plt
from inference import run_inference

images = run_inference(
    checkpoint_dir="checkpoints/step_800",
    prompt="a sks person on the moon",
    output_dir="outputs",
    device=device,
    dtype=dtype,
    num_images=4,
)

fig, axes = plt.subplots(1, len(images), figsize=(4 * len(images), 4))
for ax, img in zip(axes, images):
    ax.imshow(img)
    ax.axis("off")
plt.tight_layout()
plt.show()